# TP2: Rosenblatt's Perceptron FROM SCRATCH
**University Assignment**: Pure NumPy implementation (NO scikit-learn Perceptron class).
- Part A: Binary classification (Setosa vs Versicolor) with 2 features (Petal Length & Width).
- Part C: Multiclass One-vs-Rest (OvR) classification with 2 and 4 features.
Colab-ready notebook with error curves, decision boundary visualization, and theory Q&A.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print('NumPy and scikit-learn datasets imported successfully!')

## 1. Perceptron Class Implementation (Pure NumPy)
Mathematical formulation:
$$z = w \cdot x + b$$
$$f(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$
Update rule:
$$w \leftarrow w + \eta (y - \hat{y}) x$$
$$b \leftarrow b + \eta (y - \hat{y})$$

In [ ]:
class Perceptron:
    def __init__(self, lr=0.1, epochs=20, seed=42):
        self.lr = lr
        self.epochs = epochs
        self.seed = seed
        self.w = None
        self.b = 0.0
        self.errors_ = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.seed)
        self.w = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b = 0.0
        self.errors_ = []

        for _ in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):
                z = np.dot(xi, self.w) + self.b
                y_hat = 1 if z >= 0.0 else 0
                update = self.lr * (target - y_hat)
                if update != 0.0:
                    self.w += update * xi
                    self.b += update
                    errors += 1
            self.errors_.append(errors)
        return self

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0.0, 1, 0)

## 2. Part A: Binary Classification (Setosa vs Versicolor)
Features: `petal length` and `petal width`.
Expected: Linearly separable, 100% accuracy, training errors reach 0 within ~2 epochs.

In [ ]:
iris = load_iris()
mask = iris.target < 2
X_bin = iris.data[mask][:, [2, 3]]
y_bin = iris.target[mask]

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

p_bin = Perceptron(lr=0.1, epochs=20, seed=42)
p_bin.fit(X_tr_b, y_tr_b)
y_pred_b = p_bin.predict(X_te_b)

print('Binary Test Accuracy:', accuracy_score(y_te_b, y_pred_b))
print('Training errors per epoch:', p_bin.errors_)

# Plot training error curve
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(p_bin.errors_) + 1), p_bin.errors_, marker='o', color='crimson')
plt.xlabel('Epoch')
plt.ylabel('Misclassified Updates')
plt.title('Training Error Curve (Setosa vs Versicolor)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Part C: Multiclass Classification with One-vs-Rest (OvR)
Training 3 binary perceptrons, predicting via $\arg\max(z_k)$.

In [ ]:
# 4 features
X_all = iris.data
y_all = iris.target
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

perceptrons_4 = []
for c in range(3):
    y_c = np.where(y_tr_m == c, 1, 0)
    p = Perceptron(lr=0.1, epochs=100, seed=42).fit(X_tr_m, y_c)
    perceptrons_4.append(p)

scores_4 = np.column_stack([p.decision_function(X_te_m) for p in perceptrons_4])
y_pred_4 = np.argmax(scores_4, axis=1)
print('OvR Accuracy with all 4 features:', accuracy_score(y_te_m, y_pred_4))

# 2 features (petal length, petal width)
X_2 = iris.data[:, [2, 3]]
X_tr_2, X_te_2, y_tr_2, y_te_2 = train_test_split(
    X_2, y_all, test_size=0.2, random_state=42, stratify=y_all
)
perceptrons_2 = []
for c in range(3):
    y_c = np.where(y_tr_2 == c, 1, 0)
    p = Perceptron(lr=0.1, epochs=100, seed=42).fit(X_tr_2, y_c)
    perceptrons_2.append(p)

scores_2 = np.column_stack([p.decision_function(X_te_2) for p in perceptrons_2])
y_pred_2 = np.argmax(scores_2, axis=1)
print('OvR Accuracy with 2 features:', accuracy_score(y_te_2, y_pred_2))
print('Note: Versicolor and Virginica overlap in 2D and are NOT linearly separable, which limits 2-feature accuracy.')

## Theory Questions & Answers
1. **Why is an activation function needed?**
   - It maps the linear continuous sum $z = w \cdot x + b$ into a non-linear or thresholded decision space ($0$ or $1$) to classify samples.
2. **Role of learning rate $\eta$?**
   - Controls step size during weight updates. Too small = slow convergence; too large = oscillations and failure to converge.
3. **Can a single-layer perceptron solve XOR?**
   - No. Minsky & Papert (1969) proved it only solves linearly separable problems. XOR requires non-linear decision boundaries (Multi-Layer Perceptrons).
4. **How does One-vs-Rest (OvR) multiclass work?**
   - Trains $K$ binary models (each class vs all others). Test sample is evaluated on all $K$ models, taking the $\arg\max$ of the raw linear discriminant scores $z$.